In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import concurrent.futures


import numpy as np
import pandas as pd
import polars as pl

import ast
import glob
import pickle
import dask
import os
import itertools

import pickle

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm
from collections import Counter
from functools import reduce

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
augmented_df = pd.read_parquet("../data/imputed_augmented_us_counties.parquet")
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df = augmented_df.sort_values(by=["fips","date"])


In [ ]:
gt_columns = ["fips", "days_from_start", "shifted_log_rolled_cases"]
augmented_df_gt = augmented_df[gt_columns]
grouped = augmented_df_gt.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


### Load TLGRF Benchmark Dataset

In [ ]:
benchmark_TLGRF_dataset = pd.read_parquet("../data/benchmark_tlgrf_imputed/")
benchmark_TLGRF_dataset["date"] = pd.to_datetime(benchmark_TLGRF_dataset["date"])

df = benchmark_TLGRF_dataset.copy()
fips_list = df["fips"].unique()
display(benchmark_TLGRF_dataset)


### Read in LLF CF Results

In [ ]:
def read_csv_file(file_path):
    # Read the CSV file into a pandas DataFrame
    try:
        GRF_df = pd.read_csv(file_path)
        return GRF_df
    except pd.errors.EmptyDataError:
        print(file_path)

In [ ]:
llf_cf_results_dfs = [pd.read_parquet("../data/benchmark_llf_classical/")]


In [ ]:
llf_cf_results = pd.concat(llf_cf_results_dfs).sort_values(by=["fips", "days_from_start"])
llf_cf_results["date"] = pd.to_datetime(llf_cf_results["date"])
llf_cf_results = llf_cf_results.rename(columns={"predicted_log_rolled_cases_LLF":"LLF_predicted_log_rolled_cases"})
llf_cf_results = llf_cf_results.dropna(subset=["LLF_predicted_log_rolled_cases"])
llf_cf_results = pd.merge(llf_cf_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(llf_cf_results)

### Read in LLF TLGRF Results

In [ ]:
LLF_TLGRF_results_dfs = [pd.read_parquet("../data/benchmark_llf_tlgrf/")]


In [ ]:
LLF_TLGRF_results = pd.concat(LLF_TLGRF_results_dfs).sort_values(by=["fips", "days_from_start"])
LLF_TLGRF_results["date"] = pd.to_datetime(LLF_TLGRF_results["datetime"])
LLF_TLGRF_results = LLF_TLGRF_results.rename(columns={"predicted.grf.future.0":"LLF_TLGRF_predicted_log_rolled_cases"})
LLF_TLGRF_results = LLF_TLGRF_results.dropna(subset=["LLF_TLGRF_predicted_log_rolled_cases"])
LLF_TLGRF_results = pd.merge(LLF_TLGRF_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(LLF_TLGRF_results)

In [ ]:
LLF_TLGRF_results.columns

### Read TLGRF Top 10 Results

In [ ]:
TLGRF_Top10_results_dfs = [pd.read_parquet("../data/benchmark_tlgrf_top10/")]


In [ ]:
TLGRF_Top10_results = pd.concat(TLGRF_Top10_results_dfs).sort_values(by=["fips", "days_from_start"])
TLGRF_Top10_results["date"] = pd.to_datetime(TLGRF_Top10_results["datetime"])
TLGRF_Top10_results = TLGRF_Top10_results.rename(columns={"predicted.grf.future.0":"LLF_TLGRF_predicted_log_rolled_cases"})
TLGRF_Top10_results = TLGRF_Top10_results.dropna(subset=["LLF_TLGRF_predicted_log_rolled_cases"])
TLGRF_Top10_results = pd.merge(TLGRF_Top10_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(TLGRF_Top10_results)

### Load LL Regression Forest Blocked

In [ ]:
LL_Regression_Blocked_results_dfs = [pd.read_parquet("../data/benchmark_llrf_blocked/")]


In [ ]:
LL_Regression_Blocked_results = pd.concat(LL_Regression_Blocked_results_dfs).sort_values(by=["fips", "days_from_start"])
LL_Regression_Blocked_results["date"] = pd.to_datetime(LL_Regression_Blocked_results["date"])
LL_Regression_Blocked_results = LL_Regression_Blocked_results.rename(columns={"predicted_log_rolled_cases_LLF":"LLF_predicted_log_rolled_cases"})
LL_Regression_Blocked_results = LL_Regression_Blocked_results.dropna(subset=["LLF_predicted_log_rolled_cases"])
LL_Regression_Blocked_results = pd.merge(LL_Regression_Blocked_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(LL_Regression_Blocked_results)

### Load LL Regression Forest Blocked Top 10

In [ ]:
LL_Regression_Blocked_results_Top10_dfs = [pd.read_parquet("../data/benchmark_llrf_blocked_top10/")]


In [ ]:
LL_Regression_Blocked_Top10_results = pd.concat(LL_Regression_Blocked_results_Top10_dfs).sort_values(by=["fips", "days_from_start"])
LL_Regression_Blocked_Top10_results["date"] = pd.to_datetime(LL_Regression_Blocked_results["date"])
LL_Regression_Blocked_Top10_results = LL_Regression_Blocked_Top10_results.rename(columns={"predicted_log_rolled_cases_LLF":"LLF_predicted_log_rolled_cases"})
LL_Regression_Blocked_Top10_results = LL_Regression_Blocked_Top10_results.dropna(subset=["LLF_predicted_log_rolled_cases"])
LL_Regression_Blocked_Top10_results = pd.merge(LL_Regression_Blocked_Top10_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(LL_Regression_Blocked_Top10_results)

### Load Regression Forest Blocked

In [ ]:
Regression_Blocked_results_dfs = [pd.read_parquet("../data/benchmark_regression_forest_blocked/")]


In [ ]:
Regression_Blocked_results = pd.concat(Regression_Blocked_results_dfs).sort_values(by=["fips", "days_from_start"])
Regression_Blocked_results["date"] = pd.to_datetime(Regression_Blocked_results["date"])
Regression_Blocked_results = Regression_Blocked_results.rename(columns={"predicted_log_rolled_cases_LLF":"LLF_predicted_log_rolled_cases"})
Regression_Blocked_results = Regression_Blocked_results.dropna(subset=["LLF_predicted_log_rolled_cases"])
Regression_Blocked_results = pd.merge(Regression_Blocked_results, augmented_df_gt, on=["fips","days_from_start"], how="left")
#time_invariant_GRF_results = time_invariant_GRF_results.drop(columns=["predicted.grf.future", "predicted.grf.future.0", "Predicted_Double_Days"])
display(Regression_Blocked_results)

### Metrics

In [ ]:
rmse_LLF_func = lambda x: np.sqrt(np.nanmean((x["LLF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]) ** 2))
mae_LLF_func = lambda x: np.nanmean(np.abs(x["LLF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]))

rmse_TLGRF_func = lambda x: np.sqrt(np.nanmean((x["TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]) ** 2))
mae_TLGRF_func = lambda x: np.nanmean(np.abs(x["TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]))

rmse_LLF_TLGRF_func = lambda x: np.sqrt(np.nanmean((x["LLF_TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]) ** 2))
mae_LLF_TLGRF_func = lambda x: np.nanmean(np.abs(x["LLF_TLGRF_predicted_log_rolled_cases"] - x["shifted_log_rolled_cases"]))

log_20 = np.log(20 + 1.1)

TLGRF_df = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["log_rolled_cases"] >= log_20]
TLGRF_df = TLGRF_df[TLGRF_df["date"] <= "2022-12-31"]

LLF_CF_df = llf_cf_results[llf_cf_results["log_rolled_cases"] >= log_20]
LLF_CF_df = LLF_CF_df[LLF_CF_df["date"] <= "2022-12-31"]

LL_Regression_Blocked_df = LL_Regression_Blocked_results[LL_Regression_Blocked_results["log_rolled_cases"] >= log_20]
LL_Regression_Blocked_df = LL_Regression_Blocked_df[LL_Regression_Blocked_df["date"] <= "2022-12-31"]

LL_Regression_Blocked_Top10_df = LL_Regression_Blocked_Top10_results[LL_Regression_Blocked_Top10_results["log_rolled_cases"] >= log_20]
LL_Regression_Blocked_Top10_df = LL_Regression_Blocked_Top10_df[LL_Regression_Blocked_Top10_df["date"] <= "2022-12-31"]

Regression_Blocked_df = Regression_Blocked_results[Regression_Blocked_results["log_rolled_cases"] >= log_20]
Regression_Blocked_df = Regression_Blocked_df[Regression_Blocked_df["date"] <= "2022-12-31"]


LLF_TLGRF_df = LLF_TLGRF_results[LLF_TLGRF_results["log_rolled_cases.x"] >= log_20]
LLF_TLGRF_df = LLF_TLGRF_df[LLF_TLGRF_df["date"] <= "2022-12-31"]

TLGRF_Top10_df = TLGRF_Top10_results[TLGRF_Top10_results["log_rolled_cases.x"] >= log_20]
TLGRF_Top10_df = TLGRF_Top10_df[TLGRF_Top10_df["date"] <= "2022-12-31"]


#TLGRF
RMSE_TLGRF = TLGRF_df.groupby("date").apply(rmse_TLGRF_func)
MAE_TLGRF = TLGRF_df.groupby("date").apply(mae_TLGRF_func)
#LLF CF
RMSE_LLF_CF_df = LLF_CF_df.groupby("date").apply(rmse_LLF_func)
MAE_LLF_CF_df = LLF_CF_df.groupby("date").apply(mae_LLF_func)
#LL Regression Blocked
RMSE_LL_Regression_Blocked_df = LL_Regression_Blocked_df.groupby("date").apply(rmse_LLF_func)
MAE_LL_Regression_Blocked_df = LL_Regression_Blocked_df.groupby("date").apply(mae_LLF_func)

#LL Regression Blocked Top 10
RMSE_LL_Regression_Blocked_Top10_df = LL_Regression_Blocked_Top10_df.groupby("date").apply(rmse_LLF_func)
MAE_LL_Regression_Blocked_Top10_df = LL_Regression_Blocked_Top10_df.groupby("date").apply(mae_LLF_func)

# Regression Blocked
RMSE_Regression_Blocked_df = Regression_Blocked_df.groupby("date").apply(rmse_LLF_func)
MAE_Regression_Blocked_df = Regression_Blocked_df.groupby("date").apply(mae_LLF_func)


#LLF TLGRF
RMSE_LLF_TLGRF_df = LLF_TLGRF_df.groupby("date").apply(rmse_LLF_TLGRF_func)
MAE_LLF_TLGRF_df = LLF_TLGRF_df.groupby("date").apply(mae_LLF_TLGRF_func)

# TLGRF Top10
RMSE_TLGRF_Top10_df = TLGRF_Top10_df.groupby("date").apply(rmse_LLF_TLGRF_func)
MAE_TLGRF_Top10_df = TLGRF_Top10_df.groupby("date").apply(mae_LLF_TLGRF_func)



In [ ]:
combined_daily_metrics_df = pd.DataFrame()
combined_daily_metrics_df["MAE TLGRF"] = MAE_TLGRF
combined_daily_metrics_df["RMSE TLGRF"] = RMSE_TLGRF

combined_daily_metrics_df["MAE LLF CF"] = MAE_LLF_CF_df
combined_daily_metrics_df["RMSE LLF CF"] = RMSE_LLF_CF_df

combined_daily_metrics_df["MAE LLF TLGRF"] = MAE_LLF_TLGRF_df
combined_daily_metrics_df["RMSE LLF TLGRFF"] = RMSE_LLF_TLGRF_df

combined_daily_metrics_df["MAE TLGRF Top 10"] = MAE_TLGRF_Top10_df
combined_daily_metrics_df["RMSE TLGRF Top 10"] = RMSE_TLGRF_Top10_df

combined_daily_metrics_df["MAE LL Regression Blocked"] = MAE_LL_Regression_Blocked_df
combined_daily_metrics_df["RMSE LL Regression Blocked"] = RMSE_LL_Regression_Blocked_df

combined_daily_metrics_df["MAE LL Regression Blocked Top 10"] = MAE_LL_Regression_Blocked_Top10_df
combined_daily_metrics_df["RMSE LL Regression Blocked Top 10"] = RMSE_LL_Regression_Blocked_Top10_df

combined_daily_metrics_df["MAE Regression Blocked"] = MAE_Regression_Blocked_df
combined_daily_metrics_df["RMSE Regression Blocked"] = RMSE_Regression_Blocked_df


combined_daily_metrics_df = combined_daily_metrics_df.dropna(subset=['MAE TLGRF'])



combined_daily_metrics_df.to_csv("combined_LLF_daily_metrics.csv" ,index=True)

In [ ]:
combined_daily_metrics_df

In [ ]:
combined_daily_metrics_df.columns

In [ ]:
## Now train a new TLGRF, don't call local linearity when predicting, use the Top 10 most important features
### aka TLGRF Top 10 Only

In [ ]:
metrics_comparison_df = pd.DataFrame()

metrics_comparison_df["MAE"] = [MAE_LLF_CF_df.median(), MAE_LL_Regression_Blocked_df.median(),MAE_LL_Regression_Blocked_Top10_df.median(), MAE_Regression_Blocked_df.median(), MAE_LLF_TLGRF_df.median(), MAE_TLGRF_Top10_df.median(), MAE_TLGRF.median()]
metrics_comparison_df["RMSE"] = [RMSE_LLF_CF_df.median(), RMSE_LL_Regression_Blocked_df.median(), RMSE_LL_Regression_Blocked_Top10_df.median(), RMSE_Regression_Blocked_df.median(), RMSE_LLF_TLGRF_df.median(), RMSE_TLGRF_Top10_df.median(), RMSE_TLGRF.median()]
metrics_comparison_df.index = ["LLF CF", "LL Regression Blocked", "LL Regression Blocked Top10", "Regression Blocked", "LLF TLGRF", "TLGRF Top10", "TLGRF"]
metrics_comparison_df

### Describe the experiments in three dimensions:
 * During training time, did we use all features or top 10
 * During prediction time, do we correct for the bias via local linearity on all features / on top 10 / not at all?
 * Underlying forest: Causal Forest (with or without blocking) / Regression Forest (blocking only)

### Comments:
 * `LLF CF LL on Top 10`: Running ll_predict with local linear on the Top 10 most important features on a classical time variant causal forest (Classical GRF)
 * `LL Regression Blocked, calling ll predict on all features`: Trained ll_regression_forest on all features, with blocking structure
 * `LL Regression Blocked trained only on Top 10 features`: Trained ll_regression_forest on the Top 10 most important features only, with blocking structure, and calling local linearity on all of them
 * `Regression Blocked`: Trained a regression forest on all features, using the blocking structure
 * `LL TLGRF`: Trained TLGRF on all features, then calling predict with local linearity on the Top 10 most important features
 * `TLGRF trained only on Top 10 features without calling LL`: Trained TLGRF on Top 10 most important features only. No local lienarity
 * `TLGRF`: TLGRF as usual

In [ ]:
# Run one more: Blocked structure on Regression Forest without local linearity

In [ ]:
#Classical GRF without time varying features 0.218 0.291
#Classical GRF with time varying features 0.204 0.275
#TLGRF 0.127 0.195

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(RMSE_LLF_CF_df, label="LLF CF", color="blue")
plt.plot(RMSE_LLF_TLGRF_df, label="LLF TLGRF", color="green")
plt.plot(RMSE_LL_Regression_Blocked_df, label="LL Regression Blocked", color="orange")

plt.plot(RMSE_LL_Regression_Blocked_Top10_df, label="LL Regression Blocked Top10", color="pink")

plt.plot(RMSE_Regression_Blocked_df, label="Regression Blocked", color="purple")


plt.plot(RMSE_TLGRF_Top10_df, label="TLGRF Top 10", color="black")
plt.plot(RMSE_TLGRF, label="TLGRF", color="red")



plt.legend(loc='upper right', fontsize='large')  # Adjust the bbox_to_anchor to increase the size
plt.xlabel("Date")
plt.ylabel("RMSE")
plt.title("Root Mean Square Error (RMSE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,1.0)
plt.savefig("Benchmark_LL_TLGRF_rmse.png")
plt.show()


In [ ]:
plt.figure(figsize=(16,8))

plt.plot(MAE_LLF_CF_df, label="LLF CF", color="blue")
plt.plot(MAE_LLF_TLGRF_df, label="LLF TLGRF", color="green")
plt.plot(MAE_LL_Regression_Blocked_df, label="LL Regression Blocked", color="orange")

plt.plot(MAE_LL_Regression_Blocked_Top10_df, label="LL Regression Blocked Top10", color="pink")
plt.plot(MAE_Regression_Blocked_df, label="Regression Blocked", color="purple")


plt.plot(MAE_TLGRF_Top10_df, label="TLGRF Top 10", color="black")

plt.plot(MAE_TLGRF, label="TLGRF", color="red")

plt.legend(loc='upper right', fontsize='large')  # Adjust the bbox_to_anchor to increase the size
plt.xlabel("Date")
plt.ylabel("MAE")
plt.title("Mean Absolute Error (MAE) in One-Week Ahead COVID Case Predictions")
plt.xlim(pd.to_datetime('2020-03-15'), pd.to_datetime('2023-01-01'))
plt.ylim(0,0.6)
plt.savefig("Benchmark_LL_TLGRF_mae.png")

plt.show()

In [ ]:
RMSE_TLGRF.to_csv("RMSE_TLGRF.csv")
MAE_TLGRF.to_csv("MAE_TLGRF.csv")

RMSE_LLF_CF_df.to_csv("RMSE_LLF_CF_df.csv")
MAE_LLF_CF_df.to_csv("MAE_LLF_CF_df.csv")

RMSE_LLF_TLGRF_df.to_csv("RMSE_LLF_TLGRF_df.csv")
MAE_LLF_TLGRF_df.to_csv("MAE_LLF_TLGRF_df.csv")

RMSE_TLGRF_Top10_df.to_csv("RMSE_TLGRF_Top10_df.csv")
MAE_TLGRF_Top10_df.to_csv("MAE_TLGRF_Top10_df.csv")


RMSE_LL_Regression_Blocked_df.to_csv("RMSE_LL_Regression_Blocked_df.csv")
MAE_LL_Regression_Blocked_df.to_csv("MAE_LL_Regression_Blocked_df.csv")

RMSE_LL_Regression_Blocked_Top10_df.to_csv("RMSE_LL_Regression_Blocked_Top10_df.csv")
MAE_LL_Regression_Blocked_Top10_df.to_csv("MAE_LL_Regression_Blocked_Top10_df")

RMSE_Regression_Blocked_df.to_csv("RMSE_Regression_Blocked_df.csv")
MAE_Regression_Blocked_df.to_csv("MAE_Regression_Blocked_df.csv")


In [ ]:
pass

### Compare `r` estimates

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass